In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "brauer2009apes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Braeuer_2009_braeuer-fairnesstoken-2007.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df['study_id']="brauer2009apes"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
for x,y in zip(df_name['wrong'],df_name['right']):
    df['subject'].replace(x, y, inplace=True)
    df['competitor'].replace(x, y, inplace=True)

df['dyad']=df.subject.str.cat(df.competitor, sep='_')
# df.columns

In [4]:
df['role']='focal_participant'
df = df.rename(columns={"subject": "ape"})

role_2 = []
for index, row in df.iterrows(): 
    if row['cond'] == "same":
        role_2.append("focal_participant")
    else:
        role_2.append("competitor")
df = df.assign(role_2=role_2)
df = df.rename(columns={"competitor": "ape_2"})

In [5]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

df=df[df['ape'].notna()]

In [6]:
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

In [7]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list_2.csv")
subject_list_2 = pd.read_csv(complete_path_age)   
df= df.merge(subject_list_2,left_on='participant_2', right_on='Name_2', how='left')
df.rename(columns={"Age_2": "age_in_years_2", 'cond':'condition'}, inplace=True)

In [8]:

brauer2009apes_standardized=df[[ 'study_id','participant', 'age_in_years','sex','role',
                                 'participant_2','age_in_years_2', 'sex_2','role_2', 'species','dyad', 
     'condition', 'together', 'related', 'first_pair',
       'first_experience', 'token_soon', 'token_late', 'token_not',
       'token_latency', 'w_token_soon', 'w_token_late', 'w_token_not',
       'w_token_latency', 'eat_soon', 'eat_late',
       'eat_soon_percent', 'eat_late_percent', 'point', 'point_pp', 'if_point',
       'leave', 'leave_per', 'if_leave', 'trialduration', 'if_extra_beg',
       'if_beg', 'eat_per_present_time']]


In [9]:
comp_out_path_stand = os.path.join(out_pathway, 'brauer2009apes_standardized.csv')
brauer2009apes_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =brauer2009apes_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
brauer2009apes_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'brauer2009apes_glossary.csv')
brauer2009apes_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
